In [18]:
from pathlib import Path
from datetime import datetime
import re

root_folder = Path(
    r"G:\Data\ALS TDI\wav_recordings-20241122T210040Z-001\wav_recordings"
)


def extract_recording_time(file_path: Path):
    """
    Finds a timestamp of at least 14 digits in the filename.

    Uses the first 14 digits as:
    MMDDYYYYHHMMSS

    Examples supported:
    Patient_130_01142019210851_recording.wav
    Patient_1319_0106202120150644_recording.wav
    Patient_167-01202016091300_recording.wav
    001_Patient_130_01142019210851_recording.wav
    """

    # Find digit sequences containing at least 14 digits
    digit_groups = re.findall(r"\d{14,}", file_path.stem)

    for digit_group in digit_groups:
        timestamp_text = digit_group[:14]

        try:
            return datetime.strptime(
                timestamp_text,
                "%m%d%Y%H%M%S"
            )
        except ValueError:
            continue

    return None


rename_plan = []
skipped_files = []

patient_folders = sorted(
    folder
    for folder in root_folder.iterdir()
    if folder.is_dir() and folder.name.startswith("Patient_")
)

for patient_folder in patient_folders:

    wav_files = list(patient_folder.glob("*.wav"))

    valid_files = []
    invalid_files = []

    for file_path in wav_files:
        recording_time = extract_recording_time(file_path)

        if recording_time is None:
            invalid_files.append(file_path)
        else:
            valid_files.append((file_path, recording_time))

    # Sort only valid files chronologically
    valid_files.sort(key=lambda item: item[1])

    for position, (old_file, recording_time) in enumerate(
        valid_files,
        start=1
    ):
        # Remove an existing numeric prefix, if present
        clean_name = re.sub(
            r"^\d+_",
            "",
            old_file.name
        )

        new_name = f"{position:03d}_{clean_name}"
        new_file = old_file.with_name(new_name)

        if old_file.name != new_name:
            rename_plan.append((old_file, new_file))

    skipped_files.extend(invalid_files)


print(f"Patient folders found: {len(patient_folders)}")
print(f"Files to rename: {len(rename_plan)}")
print(f"Files skipped: {len(skipped_files)}")

print("\nFirst 30 proposed changes:\n")

for old_file, new_file in rename_plan[:30]:
    print(f"{old_file.name}")
    print(f"  -> {new_file.name}")

Patient folders found: 1099
Files to rename: 20348
Files skipped: 0

First 30 proposed changes:

Patient_1000_Visit001_07282017122111_recording.wav
  -> 001_Patient_1000_Visit001_07282017122111_recording.wav
Patient_1000_Visit002_08022021111009_recording.wav
  -> 002_Patient_1000_Visit002_08022021111009_recording.wav
Patient_1000_Visit003_06142023094436_recording.wav
  -> 003_Patient_1000_Visit003_06142023094436_recording.wav
Patient_1000_Visit004_07272023092451_recording.wav
  -> 004_Patient_1000_Visit004_07272023092451_recording.wav
Patient_1000_Visit005_08072023152429_recording.wav
  -> 005_Patient_1000_Visit005_08072023152429_recording.wav
Patient_1000_Visit006_08152023114220_recording.wav
  -> 006_Patient_1000_Visit006_08152023114220_recording.wav
Patient_1000_Visit007_08312023094648_recording.wav
  -> 007_Patient_1000_Visit007_08312023094648_recording.wav
Patient_1000_Visit008_10152023151957_recording.wav
  -> 008_Patient_1000_Visit008_10152023151957_recording.wav
Patient_1000_Vi

In [19]:
print(f"Skipped files: {len(skipped_files)}\n")

for file_path in skipped_files[:100]:
    print(file_path)

Skipped files: 0



In [20]:
# Check for duplicate destination paths
target_paths = [new_file for _, new_file in rename_plan]

if len(target_paths) != len(set(target_paths)):
    raise RuntimeError(
        "Duplicate destination filenames detected. "
        "No files were renamed."
    )


# Check whether a destination is occupied by an unrelated file
source_paths = {old_file for old_file, _ in rename_plan}

conflicts = [
    new_file
    for old_file, new_file in rename_plan
    if new_file.exists() and new_file not in source_paths
]

if conflicts:
    print("Filename conflicts found:\n")

    for conflict in conflicts[:50]:
        print(conflict)

    raise RuntimeError(
        "Renaming stopped because destination files already exist."
    )


# First rename each file to a temporary name.
# This prevents collisions when numeric prefixes change.
temporary_plan = []

for number, (old_file, final_file) in enumerate(
    rename_plan,
    start=1
):
    temporary_file = old_file.with_name(
        f"__temp_sorting_{number:07d}__.wav"
    )

    old_file.rename(temporary_file)
    temporary_plan.append(
        (temporary_file, final_file)
    )


# Rename temporary files to their final names
for temporary_file, final_file in temporary_plan:
    temporary_file.rename(final_file)


print(f"Successfully renamed {len(temporary_plan)} files.")
print(f"Skipped {len(skipped_files)} unrecognized files.")

Successfully renamed 20348 files.
Skipped 0 unrecognized files.
